# 🎯 Bayesian Thinking
**Extended · Pattern Recognition for the Rest of Us**

> Classical statistics asks: "What is the probability of this data given my hypothesis?" Bayesian statistics asks the more natural question: "What is the probability of my hypothesis given this data?"

### What you'll learn
- Prior, likelihood, posterior — the three Bayesian ingredients
- Bayes' theorem applied to real problems
- Credible intervals vs confidence intervals
- Updating beliefs with new evidence
- Conjugate priors — closed-form Bayesian updates

### Time: ~50 minutes

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from scipy import stats
from scipy.stats import beta, norm, binom

## 📐 Part 1 — Bayes' Theorem

```
P(θ | data) = P(data | θ) × P(θ) / P(data)
posterior   =  likelihood  ×  prior  / evidence
```

- **Prior P(θ):** what we believe before seeing data
- **Likelihood P(data|θ):** how probable is the data given parameter θ
- **Posterior P(θ|data):** updated belief after seeing data
- **Evidence P(data):** normalizing constant (often ignored — we care about the shape)

The key insight: Bayesian inference is about **updating beliefs** with evidence. Start with a prior, observe data, get a posterior.

In [ ]:
# Classic example: estimating a coin's fairness
# Prior: we think coin is fair (Beta(10,10) — moderate belief in p=0.5)
# Data: flip 20 times, get 15 heads
# Posterior: updated belief about p (probability of heads)

np.random.seed(42)
prior_a, prior_b = 10, 10    # prior: Beta(10,10) centered at 0.5
heads, flips = 15, 20        # observed data

# Beta-Binomial conjugacy: posterior is Beta(a+heads, b+tails)
post_a = prior_a + heads
post_b = prior_b + (flips - heads)

p_range = np.linspace(0, 1, 300)
prior_dist = beta.pdf(p_range, prior_a, prior_b)
likelihood  = binom.pmf(heads, flips, p_range) * 10  # scaled for visibility
posterior   = beta.pdf(p_range, post_a, post_b)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(p_range, prior_dist, color='#888',    lw=2, ls='--', label=f'Prior: Beta({prior_a},{prior_b}) — "probably fair"')
ax.plot(p_range, likelihood,  color='#e85d20', lw=2, ls=':',  label=f'Likelihood: {heads} heads in {flips} flips (scaled)')
ax.plot(p_range, posterior,   color='#1e5fa8', lw=2.5,        label=f'Posterior: Beta({post_a},{post_b})')
ax.axvline(0.5, color='black', lw=0.8, ls='--', alpha=0.4, label='p=0.5 (fair coin)')
ax.axvline(post_a/(post_a+post_b), color='#1a7a45', lw=1.5, ls='--',
           label=f'Posterior mean = {post_a/(post_a+post_b):.2f}')
ax.set_xlabel('p = P(heads)')
ax.set_ylabel('Density')
ax.set_title('Bayesian Update: Coin Fairness\nPrior → Likelihood → Posterior')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
ci_lo, ci_hi = beta.ppf([0.025, 0.975], post_a, post_b)
print(f"\nPrior mean:     {prior_a/(prior_a+prior_b):.2f}")
print(f"MLE (freq):     {heads/flips:.2f}  (just heads/total — ignores prior)")
print(f"Posterior mean: {post_a/(post_a+post_b):.2f}  (pulled toward prior)")
print(f"95% Credible interval: [{ci_lo:.3f}, {ci_hi:.3f}]")
print("\n📌 The posterior is pulled toward 0.5 by the prior.")
print("   With more data, the posterior converges to the MLE regardless of prior.")

In [ ]:
# Show how prior strength affects the posterior
priors = [(1,1,'Flat prior (no opinion)'), (2,2,'Weak prior'), (10,10,'Moderate prior'), (50,50,'Strong prior')]
fig, axes = plt.subplots(1,4,figsize=(16,4))

for ax, (a,b,label) in zip(axes, priors):
    post_a2, post_b2 = a+heads, b+(flips-heads)
    ax.plot(p_range, beta.pdf(p_range,a,b),         color='#888',    lw=1.5, ls='--', label='Prior')
    ax.plot(p_range, beta.pdf(p_range,post_a2,post_b2), color='#1e5fa8', lw=2,    label='Posterior')
    ax.axvline(0.5, color='black', lw=0.8, ls=':', alpha=0.4)
    ax.set_title(f'{label}\nPosterior mean={post_a2/(post_a2+post_b2):.2f}', fontsize=9)
    ax.set_xlabel('p'); ax.legend(fontsize=8)

plt.suptitle(f'Effect of Prior Strength — same data ({heads}/{flips} heads)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("📌 Strong priors pull the posterior further from the data.")
print("   With enough data, all priors eventually yield the same posterior.")

In [ ]:
# Sequential updating — beliefs evolve as new data arrives
np.random.seed(7)
true_p = 0.65   # true coin bias (unknown to us)
n_flips = 50
flips_seq = np.random.binomial(1, true_p, n_flips)

a, b = 1, 1  # start with flat prior

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
checkpoints = [1,2,5,10,15,20,30,40,50]
plot_points = [1,2,5,10,20,50]

for ax, n in zip(axes.flatten(), plot_points):
    heads_so_far = flips_seq[:n].sum()
    post_a2 = a + heads_so_far
    post_b2 = b + (n - heads_so_far)
    post_mean = post_a2 / (post_a2+post_b2)
    ci = beta.ppf([0.025,0.975], post_a2, post_b2)
    ax.plot(p_range, beta.pdf(p_range, post_a2, post_b2), color='#1e5fa8', lw=2)
    ax.axvline(true_p, color='#e85d20', lw=1.5, ls='--', label=f'True p={true_p}')
    ax.fill_between(p_range, beta.pdf(p_range, post_a2, post_b2),
                    where=(p_range >= ci[0]) & (p_range <= ci[1]),
                    alpha=0.2, color='#1e5fa8', label='95% CI')
    ax.set_title(f'After {n} flips\n({heads_so_far} heads)\nMean={post_mean:.2f}', fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('Sequential Bayesian Updating — Posterior Converges to True p=0.65', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 🔄 Part 2 — Credible Intervals vs Confidence Intervals

These sound similar but mean very different things:

**Frequentist 95% Confidence Interval:** If we repeated the experiment 100 times, 95 of the resulting intervals would contain the true parameter. The parameter is fixed — we make statements about the *procedure*.

**Bayesian 95% Credible Interval:** Given the data we observed, there is a 95% probability that the parameter lies in this interval. The parameter is *random* — we make direct probability statements about it.

Most practitioners want the Bayesian interpretation but use frequentist intervals. Bayesian intervals give you what you actually want.

In [ ]:
# Compare: CI vs Credible Interval
np.random.seed(42)
n_experiments = 50
n_obs = 30
true_mu = 5.0
sigma = 2.0

fig, ax = plt.subplots(figsize=(12, 7))
freq_contains, bayes_contains = 0, 0

for i in range(n_experiments):
    data = np.random.normal(true_mu, sigma, n_obs)
    xbar = data.mean()
    se = sigma / np.sqrt(n_obs)
    
    # Frequentist CI
    ci_lo, ci_hi = xbar - 1.96*se, xbar + 1.96*se
    freq_ok = ci_lo <= true_mu <= ci_hi
    freq_contains += freq_ok
    color_f = '#1e5fa8' if freq_ok else '#e85d20'
    ax.plot([ci_lo, ci_hi], [i-0.15, i-0.15], color=color_f, lw=1.5, alpha=0.7)
    
    # Bayesian credible interval (flat prior → same as CI here, but interpretation differs)
    post_mu  = xbar  # with flat prior, posterior mean = sample mean
    post_std = se
    cr_lo, cr_hi = norm.ppf([0.025,0.975], post_mu, post_std)
    bayes_ok = cr_lo <= true_mu <= cr_hi
    bayes_contains += bayes_ok

ax.axvline(true_mu, color='black', lw=2, label=f'True μ = {true_mu}')
ax.set_xlabel('μ')
ax.set_yticks([])
ax.set_title(f'50 Experiments — Frequentist 95% CI\nBlue = contains true μ | Red = misses | {freq_contains}/50 contain true μ')
ax.legend()
plt.tight_layout()
plt.show()
print("\n📌 Frequentist: the 95% refers to the long-run coverage of the PROCEDURE")
print("   Bayesian: the 95% refers to our BELIEF about this specific interval")

In [ ]:
answers = {
    "q1": "",  # What are the three components of Bayes' theorem?
    "q2": "",  # What is a prior distribution and where does it come from?
    "q3": "",  # What is the key difference between a 95% CI and a 95% credible interval?
    "q4": "",  # Why does a strong prior have less influence when n is large?
    "q5": "",  # Name one practical situation where a Bayesian approach is more natural than frequentist
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Bayesian Thinking"
# @title 🤖 AI-Graded Submission — fill in the box and click ▶ Run
# @markdown ---
# @markdown **Step 1:** Enter your GitHub username below
# @markdown
# @markdown **Step 2 (one-time):** Get a free AI grading key
# @markdown - Go to [aistudio.google.com](https://aistudio.google.com) — use your **personal Gmail** (not university email — many universities block AI Studio)
# @markdown - Click **Get API key → Create API key** → copy it
# @markdown - In Colab: click the **🔑 key icon** in the left sidebar → Add secret → Name: `GEMINI_API_KEY` → paste key → toggle ON
# @markdown - Done — this persists across all 30 notebooks automatically
# @markdown
# @markdown **Step 3:** Click ▶ Run on this cell for instant AI feedback

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

# ── DO NOT EDIT BELOW THIS LINE ──────────────────────────────────────────────
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Try to explain your reasoning in 1-2 sentences."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback on your answers."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with good detail. "
                             "Add a free Gemini key for AI analysis of your specific reasoning."),
                "tip": "Get a free key at aistudio.google.com using your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             f"Complete the remaining {n_total - n_answered} questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Enter your GitHub username in the box above")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"\n  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    print(f"  Student  : @{username}" if username else
          "  Student  : \u26a0\ufe0f  Enter your GitHub username in the box above")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...\n")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)\n")
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell\n")
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 choose your fork\n")


---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*